# Day 11 · Exercise 5: Search the Index

**What you'll build:** `semantic_search(query: str, index: list[dict], model: str, top_k: int) -> list[dict]` — a function that embeds the query with Ollama, computes cosine similarity against every pre-built index entry, sorts results descending by score, and returns the top-k matches as dicts with `text` and `score` keys.

**Why it matters:** This is the query-time half of semantic search — once you can rank a corpus against an arbitrary question by meaning rather than keywords, you have the retrieval engine at the heart of every RAG system you will ever build.

## Your Implementation

In [ ]:
import math
import ollama


# ── Helpers already provided — do NOT modify ───────────────────────────────

def _dot(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


def _magnitude(v: list[float]) -> float:
    return math.sqrt(sum(x * x for x in v))


def _cosine_similarity(a: list[float], b: list[float]) -> float:
    mag_a = _magnitude(a)
    mag_b = _magnitude(b)
    if mag_a == 0.0 or mag_b == 0.0:
        return 0.0
    return _dot(a, b) / (mag_a * mag_b)

# ───────────────────────────────────────────────────────────────────────────


def semantic_search(
    query: str,
    index: list[dict],
    model: str,
    top_k: int,
) -> list[dict]:
    """Embed the query, score every index entry by cosine similarity, and return the top-k results.

    The index is a list of dicts produced by `build_index`; each dict has at
    least two keys: 'text' (str) and 'embedding' (list[float]).  This function
    performs only ONE model call (to embed the query) — the document embeddings
    are already stored in the index.

    Args:
        query:  The natural-language search string to embed and score against.
        index:  Pre-built list of dicts, each with 'text' and 'embedding' keys.
        model:  Ollama embedding model name (e.g. 'nomic-embed-text').
        top_k:  Maximum number of results to return, sorted best-first.

    Returns:
        A list of dicts (length <= top_k), sorted descending by similarity.
        Each dict has two keys:
          - 'text'  (str):   the original indexed text chunk
          - 'score' (float): cosine similarity with the query vector

    Example:
        >>> index = [{"text": "Python is a language", "embedding": [0.1, 0.2]},
        ...          {"text": "Dogs are mammals",     "embedding": [0.9, 0.0]}]
        >>> results = semantic_search("programming", index, "nomic-embed-text", top_k=1)
        >>> results[0]["text"]
        'Python is a language'
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
import math
import inspect

_PASS, _FAIL = '✅', '❌'

# ── Tiny synthetic index for checks that do not call Ollama ────────────────
# Three entries with hand-crafted embeddings in 2-D space.
# Entry 0 points in the (1, 0) direction — aligned with query_vec_a.
# Entry 1 points in the (0, 1) direction — orthogonal to query_vec_a.
# Entry 2 points in the (-1, 0) direction — opposite to query_vec_a.
_SYNTHETIC_INDEX = [
    {"text": "entry A", "embedding": [1.0, 0.0]},
    {"text": "entry B", "embedding": [0.0, 1.0]},
    {"text": "entry C", "embedding": [-1.0, 0.0]},
]

# Query vector that points in the (1, 0) direction — identical to entry A.
_QUERY_VEC_A = [1.0, 0.0]


def _run_checks():
    score, total = 0, 4

    # ── Check 1: function exists and is a coroutine function ──────────────
    try:
        assert callable(semantic_search), 'semantic_search is not defined'
        print(f'{_PASS} Check 1/{total}: semantic_search is defined and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # ── Check 2: results are sorted descending by score ───────────────────
    # Monkey-patch ollama.embeddings so this check runs without Ollama.
    import ollama as _ollama
    _real_embeddings = _ollama.embeddings

    def _fake_embeddings(model, prompt):
        # Always return the query vector [1, 0] regardless of prompt.
        return {"embedding": list(_QUERY_VEC_A)}

    _ollama.embeddings = _fake_embeddings
    try:
        results = semantic_search(
            query="ignored",
            index=_SYNTHETIC_INDEX,
            model="nomic-embed-text",
            top_k=3,
        )
        assert isinstance(results, list), (
            f'expected list, got {type(results).__name__}'
        )
        assert len(results) == 3, (
            f'top_k=3 should return 3 results, got {len(results)}'
        )
        scores = [r["score"] for r in results]
        assert scores == sorted(scores, reverse=True), (
            f'results not sorted descending: {scores}'
        )
        print(f'{_PASS} Check 2/{total}: results are sorted descending by score')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
    finally:
        _ollama.embeddings = _real_embeddings

    # ── Check 3: top result is entry A (score ≈ 1.0) and has required keys ─
    _ollama.embeddings = _fake_embeddings
    try:
        results = semantic_search(
            query="ignored",
            index=_SYNTHETIC_INDEX,
            model="nomic-embed-text",
            top_k=1,
        )
        assert len(results) == 1, (
            f'top_k=1 should return 1 result, got {len(results)}'
        )
        top = results[0]
        assert "text" in top and "score" in top, (
            f'each result dict must have "text" and "score" keys; got {list(top.keys())}'
        )
        assert top["text"] == "entry A", (
            f'expected "entry A" as top result, got {top["text"]!r}'
        )
        assert abs(top["score"] - 1.0) < 1e-9, (
            f'expected score ≈ 1.0 for identical direction, got {top["score"]}'
        )
        print(f'{_PASS} Check 3/{total}: top result is correct (entry A, score ≈ 1.0)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')
    finally:
        _ollama.embeddings = _real_embeddings

    # ── Check 4: top_k clips the result list length ────────────────────────
    _ollama.embeddings = _fake_embeddings
    try:
        results_2 = semantic_search(
            query="ignored",
            index=_SYNTHETIC_INDEX,
            model="nomic-embed-text",
            top_k=2,
        )
        assert len(results_2) == 2, (
            f'top_k=2 should return 2 results, got {len(results_2)}'
        )
        results_1 = semantic_search(
            query="ignored",
            index=_SYNTHETIC_INDEX,
            model="nomic-embed-text",
            top_k=1,
        )
        assert len(results_1) == 1, (
            f'top_k=1 should return 1 result, got {len(results_1)}'
        )
        print(f'{_PASS} Check 4/{total}: top_k correctly clips result list length')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')
    finally:
        _ollama.embeddings = _real_embeddings

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Extend `semantic_search` with a `min_score` guard: if the top result's score is below a threshold (e.g. `0.35`), return an empty list rather than low-confidence results.

This foreshadows Day 11's project, where you will wire that guard into a full RAG pipeline to prevent the language model from hallucinating answers when the corpus does not contain relevant information — a single `if`-statement that turns a fragile demo into a production-safe system.

```python
def semantic_search_safe(
    query: str,
    index: list[dict],
    model: str,
    top_k: int,
    min_score: float = 0.35,
) -> list[dict]:
    results = semantic_search(query, index, model, top_k)
    if not results or results[0]["score"] < min_score:
        return []  # Caller can detect this and refuse to generate
    return results
```

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import math
import ollama


def _dot(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


def _magnitude(v: list[float]) -> float:
    return math.sqrt(sum(x * x for x in v))


def _cosine_similarity(a: list[float], b: list[float]) -> float:
    mag_a = _magnitude(a)
    mag_b = _magnitude(b)
    if mag_a == 0.0 or mag_b == 0.0:
        return 0.0
    return _dot(a, b) / (mag_a * mag_b)


def semantic_search(
    query: str,
    index: list[dict],
    model: str,
    top_k: int,
) -> list[dict]:
    """Embed the query, score every index entry by cosine similarity, and return the top-k results."""
    response = ollama.embeddings(model=model, prompt=query)
    query_vec = response["embedding"]

    scored = [
        {"text": entry["text"], "score": _cosine_similarity(query_vec, entry["embedding"])}
        for entry in index
    ]

    scored.sort(key=lambda e: e["score"], reverse=True)
    return scored[:top_k]
```

**Why this works:** The function makes exactly one `ollama.embeddings` call — for the query — because the document embeddings were already computed at index-build time. Scoring is a list comprehension over all entries, each computing a cosine similarity against the query vector. `sorted(..., reverse=True)` puts the highest-similarity entry at index 0, and slicing to `top_k` limits the result to the requested depth. The whole operation is O(n) in corpus size, which is fast enough for tens of thousands of entries in pure Python.
</details>